In [24]:
# ============================================================
# 1. IMPORT LIBRARIES AND CONFIGURE THE TRANSFORMATION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

# Use UTC consistently for healthcare transaction timestamps
spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

# Project configuration
BRONZE_PATH = "Files/bronze/synthea"
SILVER_SCHEMA = "silver"
SOURCE_SYSTEM = "Synthea CSV November 2021"

SOURCE_FILE = "claims_transactions.csv"
SOURCE_TABLE = "claims_transactions"
TARGET_TABLE = f"{SILVER_SCHEMA}.claims_transactions"

# Create the Silver schema if it does not already exist
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}"
)

print("Configuration completed.")
print(f"Bronze source: {BRONZE_PATH}")
print(f"Source file: {SOURCE_FILE}")
print(f"Target table: {TARGET_TABLE}")


StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 26, Finished, Available, Finished, False)

Configuration completed.
Bronze source: Files/bronze/synthea
Source file: claims_transactions.csv
Target table: silver.claims_transactions


In [25]:
# ============================================================
# 2. INGEST THE RAW BRONZE CLAIM-TRANSACTIONS CSV FILE
# ============================================================

claim_transaction_file_path = (
    f"{BRONZE_PATH}/{SOURCE_FILE}"
)

claims_transactions_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(
        claim_transaction_file_path
    )
)

raw_claim_transaction_count = (
    claims_transactions_raw.count()
)

print(
    f"Source table: {SOURCE_TABLE}"
)

print(
    f"Source file: {claim_transaction_file_path}"
)

print(
    f"Rows loaded: "
    f"{raw_claim_transaction_count:,}"
)

print(
    f"Columns loaded: "
    f"{len(claims_transactions_raw.columns)}"
)

claims_transactions_raw.printSchema()

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 27, Finished, Available, Finished, False)

Source table: claims_transactions
Source file: Files/bronze/synthea/claims_transactions.csv
Rows loaded: 711,238
Columns loaded: 33
root
 |-- ID: string (nullable = true)
 |-- CLAIMID: string (nullable = true)
 |-- CHARGEID: string (nullable = true)
 |-- PATIENTID: string (nullable = true)
 |-- TYPE: string (nullable = true)
 |-- AMOUNT: string (nullable = true)
 |-- METHOD: string (nullable = true)
 |-- FROMDATE: string (nullable = true)
 |-- TODATE: string (nullable = true)
 |-- PLACEOFSERVICE: string (nullable = true)
 |-- PROCEDURECODE: string (nullable = true)
 |-- MODIFIER1: string (nullable = true)
 |-- MODIFIER2: string (nullable = true)
 |-- DIAGNOSISREF1: string (nullable = true)
 |-- DIAGNOSISREF2: string (nullable = true)
 |-- DIAGNOSISREF3: string (nullable = true)
 |-- DIAGNOSISREF4: string (nullable = true)
 |-- UNITS: string (nullable = true)
 |-- DEPARTMENTID: string (nullable = true)
 |-- NOTES: string (nullable = true)
 |-- UNITAMOUNT: string (nullable = true)
 |-- T

In [26]:
# ============================================================
# 3. INSPECT RAW CLAIM-TRANSACTION RECORDS
# ============================================================

display(
    claims_transactions_raw.limit(10)
)

print(
    claims_transactions_raw.columns
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c4492a2-d2e8-43bd-a94a-e12650d7bcb9)

['ID', 'CLAIMID', 'CHARGEID', 'PATIENTID', 'TYPE', 'AMOUNT', 'METHOD', 'FROMDATE', 'TODATE', 'PLACEOFSERVICE', 'PROCEDURECODE', 'MODIFIER1', 'MODIFIER2', 'DIAGNOSISREF1', 'DIAGNOSISREF2', 'DIAGNOSISREF3', 'DIAGNOSISREF4', 'UNITS', 'DEPARTMENTID', 'NOTES', 'UNITAMOUNT', 'TRANSFEROUTID', 'TRANSFERTYPE', 'PAYMENTS', 'ADJUSTMENTS', 'TRANSFERS', 'OUTSTANDING', 'APPOINTMENTID', 'LINENOTE', 'PATIENTINSURANCEID', 'FEESCHEDULEID', 'PROVIDERID', 'SUPERVISINGPROVIDERID']


In [27]:
# ============================================================
# 4. STANDARDIZE CLAIM-TRANSACTION COLUMN NAMES
# ============================================================

claim_transaction_column_mapping = {
    "ID": "transaction_id",
    "CLAIMID": "claim_id",
    "CHARGEID": "charge_id",
    "PATIENTID": "patient_id",
    "TYPE": "transaction_type",

    # General transaction or charge amount
    "AMOUNT": "transaction_amount",

    "METHOD": "payment_method",
    "FROMDATE": "service_start_timestamp",
    "TODATE": "service_end_timestamp",
    "PLACEOFSERVICE": "place_of_service",
    "PROCEDURECODE": "procedure_code",
    "MODIFIER1": "modifier_code_1",
    "MODIFIER2": "modifier_code_2",
    "DIAGNOSISREF1": "diagnosis_reference_1",
    "DIAGNOSISREF2": "diagnosis_reference_2",
    "DIAGNOSISREF3": "diagnosis_reference_3",
    "DIAGNOSISREF4": "diagnosis_reference_4",
    "UNITS": "units",
    "DEPARTMENTID": "department_id",
    "NOTES": "transaction_notes",
    "UNITAMOUNT": "unit_amount",
    "TRANSFEROUTID": "transfer_out_id",
    "TRANSFERTYPE": "transfer_type",

    # Actual payment value must come from PAYMENTS
    "PAYMENTS": "payment_amount",

    "ADJUSTMENTS": "adjustment_amount",
    "TRANSFERS": "transfer_amount",
    "OUTSTANDING": "outstanding_amount",

    # Encounter relationship
    "APPOINTMENTID": "encounter_id",

    "LINENOTE": "line_note",

    # Preserve as a source anomaly, not as a payer key
    "PATIENTINSURANCEID":
        "source_patient_insurance_id",

    "FEESCHEDULEID": "fee_schedule_id",
    "PROVIDERID": "provider_id",
    "SUPERVISINGPROVIDERID":
        "supervising_provider_id"
}

claims_transactions_standardized = (
    claims_transactions_raw
)

for old_name in claims_transactions_raw.columns:

    if old_name in claim_transaction_column_mapping:
        new_name = claim_transaction_column_mapping[
            old_name
        ]
    else:
        new_name = (
            old_name
            .strip()
            .lower()
        )

    claims_transactions_standardized = (
        claims_transactions_standardized
        .withColumnRenamed(
            old_name,
            new_name
        )
    )

# Stop if renaming creates duplicate column names
if (
    len(claims_transactions_standardized.columns)
    !=
    len(set(claims_transactions_standardized.columns))
):
    raise ValueError(
        "Duplicate column names were created while "
        "standardizing claim transactions."
    )

print(
    f"Claim transactions: "
    f"{len(claims_transactions_raw.columns)} source columns -> "
    f"{len(claims_transactions_standardized.columns)} "
    f"standardized columns"
)

print(
    claims_transactions_standardized.columns
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 29, Finished, Available, Finished, False)

Claim transactions: 33 source columns -> 33 standardized columns
['transaction_id', 'claim_id', 'charge_id', 'patient_id', 'transaction_type', 'transaction_amount', 'payment_method', 'service_start_timestamp', 'service_end_timestamp', 'place_of_service', 'procedure_code', 'modifier_code_1', 'modifier_code_2', 'diagnosis_reference_1', 'diagnosis_reference_2', 'diagnosis_reference_3', 'diagnosis_reference_4', 'units', 'department_id', 'transaction_notes', 'unit_amount', 'transfer_out_id', 'transfer_type', 'payment_amount', 'adjustment_amount', 'transfer_amount', 'outstanding_amount', 'encounter_id', 'line_note', 'source_patient_insurance_id', 'fee_schedule_id', 'provider_id', 'supervising_provider_id']


In [28]:
# ============================================================
# 5. CLEAN CLAIM-TRANSACTION STRING VALUES
# ============================================================

def clean_transaction_string_values(df):

    cleaned_df = df

    for column_name in df.columns:
        cleaned_df = (
            cleaned_df
            .withColumn(
                column_name,
                F.when(
                    F.trim(
                        F.col(column_name)
                    ) == "",
                    F.lit(None)
                )
                .otherwise(
                    F.trim(
                        F.col(column_name)
                    )
                )
            )
        )

    return cleaned_df


# Count records before cleaning
transaction_count_before_cleaning = (
    claims_transactions_standardized.count()
)

# Trim values and convert blank strings to null
claims_transactions_cleaned = (
    clean_transaction_string_values(
        claims_transactions_standardized
    )
)

# Count records after cleaning
transaction_count_after_cleaning = (
    claims_transactions_cleaned.count()
)

print(
    f"Claim transactions: "
    f"{transaction_count_before_cleaning:,} rows "
    f"before cleaning, "
    f"{transaction_count_after_cleaning:,} rows "
    f"after cleaning"
)

# Cleaning must not add or remove records
if (
    transaction_count_before_cleaning
    !=
    transaction_count_after_cleaning
):
    raise ValueError(
        "Claim-transaction row count changed during "
        "string-value cleaning."
    )

print(
    "Claim-transaction string cleaning and "
    "row-count reconciliation passed."
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 30, Finished, Available, Finished, False)

Claim transactions: 711,238 rows before cleaning, 711,238 rows after cleaning
Claim-transaction string cleaning and row-count reconciliation passed.


In [29]:
# ============================================================
# 6. TRANSFORM CLAIM-TRANSACTION DATA TYPES
# ============================================================

claims_transactions_silver = (
    claims_transactions_cleaned

    # Preserve the original end-date value for auditing
    .withColumn(
        "source_service_end_value",
        F.col("service_end_timestamp")
    )

    # Flag the Unix-epoch placeholder found in the source
    .withColumn(
        "service_end_was_epoch_sentinel",
        F.col("service_end_timestamp").startswith(
            "1970-01-01"
        )
    )

    # Convert the service-start value to a UTC timestamp
    .withColumn(
        "service_start_timestamp",
        F.to_timestamp(
            F.col("service_start_timestamp"),
            "yyyy-MM-dd'T'HH:mm:ssX"
        )
    )

    # Convert the 1970 placeholder to null;
    # otherwise parse the actual end timestamp
    .withColumn(
        "service_end_timestamp",
        F.when(
            F.col(
                "source_service_end_value"
            ).startswith("1970-01-01"),
            F.lit(None).cast("timestamp")
        ).otherwise(
            F.to_timestamp(
                F.col("source_service_end_value"),
                "yyyy-MM-dd'T'HH:mm:ssX"
            )
        )
    )

    # Convert financial values to fixed-precision decimals
    .withColumn(
        "transaction_amount",
        F.col("transaction_amount").cast(
            T.DecimalType(18, 2)
        )
    )
    .withColumn(
        "unit_amount",
        F.col("unit_amount").cast(
            T.DecimalType(18, 2)
        )
    )

    # payment_amount comes strictly from source PAYMENTS
    .withColumn(
        "payment_amount",
        F.col("payment_amount").cast(
            T.DecimalType(18, 2)
        )
    )
    .withColumn(
        "adjustment_amount",
        F.col("adjustment_amount").cast(
            T.DecimalType(18, 2)
        )
    )
    .withColumn(
        "transfer_amount",
        F.col("transfer_amount").cast(
            T.DecimalType(18, 2)
        )
    )
    .withColumn(
        "outstanding_amount",
        F.col("outstanding_amount").cast(
            T.DecimalType(18, 2)
        )
    )

    # Claim units can potentially be fractional
    .withColumn(
        "units",
        F.col("units").cast(
            T.DecimalType(18, 4)
        )
    )

    # Standardize descriptive transaction values
    .withColumn(
        "transaction_type",
        F.upper(
            F.col("transaction_type")
        )
    )
    .withColumn(
        "payment_method",
        F.upper(
            F.col("payment_method")
        )
    )
    .withColumn(
        "transfer_type",
        F.upper(
            F.col("transfer_type")
        )
    )

    # Create reporting fields
    .withColumn(
        "service_date",
        F.to_date(
            F.col("service_start_timestamp")
        )
    )
    .withColumn(
        "service_duration_minutes",
        F.when(
            F.col("service_end_timestamp").isNotNull(),
            (
                F.col("service_end_timestamp").cast("long")
                -
                F.col("service_start_timestamp").cast("long")
            ) / 60.0
        ).otherwise(
            F.lit(None).cast("double")
        )
    )

    # Identify payment transaction rows
    .withColumn(
        "is_payment_transaction",
        F.col("transaction_type") == "PAYMENT"
    )

    # Explicitly document the source anomaly
    .withColumn(
        "patient_insurance_id_relationship_status",
        F.lit(
            "source anomaly - not validated as payer relationship"
        )
    )
)

claims_transactions_silver.printSchema()

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 31, Finished, Available, Finished, False)

root
 |-- transaction_id: string (nullable = true)
 |-- claim_id: string (nullable = true)
 |-- charge_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- transaction_amount: decimal(18,2) (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- service_start_timestamp: timestamp (nullable = true)
 |-- service_end_timestamp: timestamp (nullable = true)
 |-- place_of_service: string (nullable = true)
 |-- procedure_code: string (nullable = true)
 |-- modifier_code_1: string (nullable = true)
 |-- modifier_code_2: string (nullable = true)
 |-- diagnosis_reference_1: string (nullable = true)
 |-- diagnosis_reference_2: string (nullable = true)
 |-- diagnosis_reference_3: string (nullable = true)
 |-- diagnosis_reference_4: string (nullable = true)
 |-- units: decimal(18,4) (nullable = true)
 |-- department_id: string (nullable = true)
 |-- transaction_notes: string (nullable = true)
 |-- unit_amount: decim

In [30]:
# ============================================================
# 7. PROFILE CLAIM-TRANSACTION DATA QUALITY
# ============================================================

transaction_quality_summary = (
    claims_transactions_silver
    .agg(
        F.count("*").alias(
            "row_count"
        ),

        F.countDistinct(
            "transaction_id"
        ).alias(
            "distinct_transaction_ids"
        ),

        F.countDistinct(
            "claim_id"
        ).alias(
            "distinct_claim_ids"
        ),

        # Required identifier checks
        F.sum(
            F.when(
                F.col("transaction_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_transaction_ids"
        ),

        F.sum(
            F.when(
                F.col("claim_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_claim_ids"
        ),

        F.sum(
            F.when(
                F.col("patient_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_patient_ids"
        ),

        F.sum(
            F.when(
                F.col("encounter_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_encounter_ids"
        ),

        # Timestamp checks
        F.sum(
            F.when(
                F.col("service_start_timestamp").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_service_start_timestamps"
        ),

        F.sum(
            F.when(
                F.col(
                    "service_end_was_epoch_sentinel"
                ) == True,
                1
            ).otherwise(0)
        ).alias(
            "epoch_end_timestamp_sentinels"
        ),

        F.sum(
            F.when(
                F.col("source_service_end_value").isNotNull()
                &
                (
                    F.col(
                        "service_end_was_epoch_sentinel"
                    ) == False
                )
                &
                F.col("service_end_timestamp").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "unparseable_non_sentinel_end_timestamps"
        ),

        F.sum(
            F.when(
                F.col("service_end_timestamp")
                <
                F.col("service_start_timestamp"),
                1
            ).otherwise(0)
        ).alias(
            "service_end_before_start"
        ),

        # Payment-specific checks
        F.sum(
            F.when(
                F.col("is_payment_transaction") == True,
                1
            ).otherwise(0)
        ).alias(
            "payment_transaction_rows"
        ),

        F.sum(
            F.when(
                (
                    F.col("is_payment_transaction") == True
                )
                &
                F.col("payment_amount").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "payment_rows_with_null_payment_amount"
        ),

        # Financial checks
        F.sum(
            F.when(
                F.col("transaction_amount") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_transaction_amounts"
        ),

        F.sum(
            F.when(
                F.col("payment_amount") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_payment_amounts"
        ),

        F.sum(
            F.when(
                F.col("adjustment_amount") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_adjustment_amounts"
        ),

        F.sum(
            F.when(
                F.col("transfer_amount") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_transfer_amounts"
        ),

        F.sum(
            F.when(
                F.col("outstanding_amount") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_outstanding_amounts"
        )
    )
)

display(
    transaction_quality_summary
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 437ffbaa-f0a9-4674-a138-886de2171f56)

In [31]:
# ============================================================
# 7.1 INVESTIGATE SERVICE END BEFORE START
# ============================================================

invalid_transaction_intervals = (
    claims_transactions_silver
    .filter(
        F.col("service_end_timestamp")
        <
        F.col("service_start_timestamp")
    )
    .select(
        "transaction_id",
        "claim_id",
        "transaction_type",
        "service_start_timestamp",
        "service_end_timestamp",
        "source_service_end_value",
        "service_duration_minutes",
        "procedure_code",
        "transaction_amount",
        "payment_amount"
    )
    .orderBy(
        "service_start_timestamp"
    )
)

invalid_transaction_interval_count = (
    invalid_transaction_intervals.count()
)

print(
    f"Transactions ending before they start: "
    f"{invalid_transaction_interval_count:,}"
)

display(
    invalid_transaction_intervals
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 33, Finished, Available, Finished, False)

Transactions ending before they start: 22


SynapseWidget(Synapse.DataFrame, da1e9540-0f97-4ff4-ae8e-24db2e92890a)

In [32]:
# ============================================================
# 7.2 HANDLE INVALID SERVICE INTERVALS
# ============================================================

claims_transactions_silver = (
    claims_transactions_silver

    # Preserve the parsed end timestamp before normalization
    .withColumn(
        "source_parsed_service_end_timestamp",
        F.col("service_end_timestamp")
    )

    # Assign an explicit data-quality classification
    .withColumn(
        "service_interval_quality",
        F.when(
            F.col("service_end_was_epoch_sentinel") == True,
            F.lit("epoch placeholder normalized to null")
        )
        .when(
            F.col("source_service_end_value").isNull(),
            F.lit("missing source end timestamp")
        )
        .when(
            F.col("service_end_timestamp").isNull(),
            F.lit("unparseable source end timestamp")
        )
        .when(
            F.col("service_end_timestamp")
            <
            F.col("service_start_timestamp"),
            F.lit("end before start normalized to null")
        )
        .otherwise(
            F.lit("valid interval")
        )
    )

    # Null invalid end timestamps for analytical use
    .withColumn(
        "service_end_timestamp",
        F.when(
            F.col("service_end_timestamp")
            <
            F.col("service_start_timestamp"),
            F.lit(None).cast("timestamp")
        ).otherwise(
            F.col("service_end_timestamp")
        )
    )

    # Recalculate duration after quality normalization
    .withColumn(
        "service_duration_minutes",
        F.when(
            F.col("service_end_timestamp").isNotNull(),
            (
                F.col("service_end_timestamp").cast("long")
                -
                F.col("service_start_timestamp").cast("long")
            ) / 60.0
        ).otherwise(
            F.lit(None).cast("double")
        )
    )
)

print(
    "Invalid service intervals were retained, flagged, "
    "and normalized for analytical use."
)

display(
    claims_transactions_silver
    .groupBy(
        "service_interval_quality"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 34, Finished, Available, Finished, False)

Invalid service intervals were retained, flagged, and normalized for analytical use.


SynapseWidget(Synapse.DataFrame, 00a3e1e7-00ab-4c2e-9166-5025759dc7f7)

In [33]:
# ============================================================
# 7.3 ENFORCE CRITICAL CLAIM-TRANSACTION VALIDATIONS
# ============================================================

transaction_validation = (
    claims_transactions_silver
    .agg(
        F.count("*").alias(
            "row_count"
        ),

        F.countDistinct(
            "transaction_id"
        ).alias(
            "distinct_transaction_ids"
        ),

        F.countDistinct(
            "claim_id"
        ).alias(
            "distinct_claim_ids"
        ),

        F.sum(
            F.when(
                F.col("transaction_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_transaction_ids"
        ),

        F.sum(
            F.when(
                F.col("claim_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_claim_ids"
        ),

        F.sum(
            F.when(
                F.col("patient_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_patient_ids"
        ),

        F.sum(
            F.when(
                F.col("encounter_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_encounter_ids"
        ),

        F.sum(
            F.when(
                F.col("service_start_timestamp").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_service_start_timestamps"
        ),

        F.sum(
            F.when(
                F.col("service_end_timestamp")
                <
                F.col("service_start_timestamp"),
                1
            ).otherwise(0)
        ).alias(
            "remaining_end_before_start"
        ),

        F.sum(
            F.when(
                F.col("service_duration_minutes") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_service_durations"
        ),

        F.sum(
            F.when(
                (
                    F.col("is_payment_transaction") == True
                )
                &
                F.col("payment_amount").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "payment_rows_with_null_payment_amount"
        ),

        F.sum(
            F.when(
                F.col("service_interval_quality")
                ==
                "end before start normalized to null",
                1
            ).otherwise(0)
        ).alias(
            "documented_invalid_intervals"
        )
    )
)

display(transaction_validation)

transaction_validation_result = (
    transaction_validation.first()
)

assert transaction_validation_result["row_count"] == 711238
assert transaction_validation_result[
    "distinct_transaction_ids"
] == 711238
assert transaction_validation_result[
    "distinct_claim_ids"
] == 117889

assert transaction_validation_result[
    "null_transaction_ids"
] == 0
assert transaction_validation_result[
    "null_claim_ids"
] == 0
assert transaction_validation_result[
    "null_patient_ids"
] == 0
assert transaction_validation_result[
    "null_encounter_ids"
] == 0
assert transaction_validation_result[
    "null_service_start_timestamps"
] == 0

assert transaction_validation_result[
    "remaining_end_before_start"
] == 0
assert transaction_validation_result[
    "negative_service_durations"
] == 0
assert transaction_validation_result[
    "payment_rows_with_null_payment_amount"
] == 0
assert transaction_validation_result[
    "documented_invalid_intervals"
] == 22

print(
    "Critical claim-transaction validations passed."
)
print(
    "All 117,889 claim IDs are represented in the "
    "transaction dataset."
)
print(
    "The 22 invalid source intervals were retained with "
    "quality flags and normalized analytical end timestamps."
)
print(
    "Payment values remain sourced exclusively from PAYMENTS."
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 938707ad-225c-43c9-a82f-0460e0baacf3)

Critical claim-transaction validations passed.
All 117,889 claim IDs are represented in the transaction dataset.
The 22 invalid source intervals were retained with quality flags and normalized analytical end timestamps.
Payment values remain sourced exclusively from PAYMENTS.


In [34]:
# ============================================================
# 8. VALIDATE TRANSACTION-TO-CLAIM RELATIONSHIPS
# ============================================================

valid_claims = (
    spark.table(
        f"{SILVER_SCHEMA}.claims"
    )
    .select(
        F.col("claim_id").alias(
            "valid_claim_id"
        )
    )
    .distinct()
)

# Find transactions whose claim_id does not exist
# in the Silver claims table
invalid_transaction_claims = (
    claims_transactions_silver.alias("transactions")
    .select(
        "transaction_id",
        "claim_id"
    )
    .join(
        valid_claims.alias("claims"),
        F.col("transactions.claim_id")
        ==
        F.col("claims.valid_claim_id"),
        "left_anti"
    )
)

invalid_transaction_claim_count = (
    invalid_transaction_claims.count()
)

# Find claims that do not have any transactions
claims_with_transactions = (
    claims_transactions_silver
    .select(
        F.col("claim_id").alias(
            "transaction_claim_id"
        )
    )
    .distinct()
)

claims_without_transactions = (
    valid_claims.alias("claims")
    .join(
        claims_with_transactions.alias("transactions"),
        F.col("claims.valid_claim_id")
        ==
        F.col("transactions.transaction_claim_id"),
        "left_anti"
    )
)

claims_without_transaction_count = (
    claims_without_transactions.count()
)

print(
    f"Transactions with an invalid claim ID: "
    f"{invalid_transaction_claim_count:,}"
)

print(
    f"Claims without a transaction: "
    f"{claims_without_transaction_count:,}"
)

if invalid_transaction_claim_count > 0:
    display(
        invalid_transaction_claims.limit(20)
    )

    raise ValueError(
        "Transaction-to-claim foreign-key validation failed."
    )

if claims_without_transaction_count > 0:
    display(
        claims_without_transactions.limit(20)
    )

    raise ValueError(
        "One or more claims do not have a transaction."
    )

print(
    "Transaction-to-claim foreign-key validation passed."
)

print(
    "Every claim has at least one transaction."
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 36, Finished, Available, Finished, False)

Transactions with an invalid claim ID: 0
Claims without a transaction: 0
Transaction-to-claim foreign-key validation passed.
Every claim has at least one transaction.


In [35]:
# ============================================================
# 8.1 VALIDATE TRANSACTION-TO-PATIENT RELATIONSHIPS
# ============================================================

valid_patients = (
    spark.table(
        f"{SILVER_SCHEMA}.patients"
    )
    .select(
        F.col("id").alias(
            "valid_patient_id"
        )
    )
    .distinct()
)

invalid_transaction_patients = (
    claims_transactions_silver.alias("transactions")
    .select(
        "transaction_id",
        "patient_id"
    )
    .join(
        valid_patients.alias("patients"),
        F.col("transactions.patient_id")
        ==
        F.col("patients.valid_patient_id"),
        "left_anti"
    )
)

invalid_transaction_patient_count = (
    invalid_transaction_patients.count()
)

print(
    f"Transactions with an invalid patient ID: "
    f"{invalid_transaction_patient_count:,}"
)

if invalid_transaction_patient_count > 0:
    display(
        invalid_transaction_patients.limit(20)
    )

    raise ValueError(
        "Transaction-to-patient foreign-key "
        "validation failed."
    )

print(
    "Transaction-to-patient foreign-key "
    "validation passed."
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 37, Finished, Available, Finished, False)

Transactions with an invalid patient ID: 0
Transaction-to-patient foreign-key validation passed.


In [36]:
# ============================================================
# 8.2 VALIDATE TRANSACTION-TO-ENCOUNTER RELATIONSHIPS
# ============================================================

valid_encounters = (
    spark.table(
        f"{SILVER_SCHEMA}.encounters"
    )
    .select(
        F.col("encounter_id").alias(
            "valid_encounter_id"
        )
    )
    .distinct()
)

invalid_transaction_encounters = (
    claims_transactions_silver.alias("transactions")
    .select(
        "transaction_id",
        "encounter_id"
    )
    .join(
        valid_encounters.alias("encounters"),
        F.col("transactions.encounter_id")
        ==
        F.col("encounters.valid_encounter_id"),
        "left_anti"
    )
)

invalid_transaction_encounter_count = (
    invalid_transaction_encounters.count()
)

print(
    f"Transactions with an invalid encounter ID: "
    f"{invalid_transaction_encounter_count:,}"
)

if invalid_transaction_encounter_count > 0:
    display(
        invalid_transaction_encounters.limit(20)
    )

    raise ValueError(
        "Transaction-to-encounter foreign-key "
        "validation failed."
    )

print(
    "Transaction-to-encounter foreign-key "
    "validation passed."
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 38, Finished, Available, Finished, False)

Transactions with an invalid encounter ID: 0
Transaction-to-encounter foreign-key validation passed.


In [37]:
# ============================================================
# 8.3 DOCUMENT PATIENTINSURANCEID SOURCE ANOMALY
# ============================================================

patient_insurance_anomaly_summary = (
    claims_transactions_silver
    .agg(
        F.count("*").alias(
            "total_transactions"
        ),

        F.sum(
            F.when(
                F.col(
                    "source_patient_insurance_id"
                ).isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_source_patient_insurance_ids"
        ),

        F.countDistinct(
            "source_patient_insurance_id"
        ).alias(
            "distinct_source_patient_insurance_ids"
        )
    )
)

display(
    patient_insurance_anomaly_summary
)

print(
    "Most frequent source PATIENTINSURANCEID values:"
)

display(
    claims_transactions_silver
    .groupBy(
        "source_patient_insurance_id"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
    .limit(20)
)

print(
    "PATIENTINSURANCEID was preserved as "
    "source_patient_insurance_id for auditing."
)

print(
    "It is a documented source anomaly and was not "
    "validated or modeled as a payer relationship."
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4f4f290a-a894-4409-9d70-e25f97dc41c8)

Most frequent source PATIENTINSURANCEID values:


SynapseWidget(Synapse.DataFrame, ddfedd75-01b7-434f-9a48-8c99532f1d1a)

PATIENTINSURANCEID was preserved as source_patient_insurance_id for auditing.
It is a documented source anomaly and was not validated or modeled as a payer relationship.


In [38]:
# ============================================================
# 9. VALIDATE DUPLICATES AND TRANSACTION PRIMARY KEY
# ============================================================

# Check for exact duplicate source records
exact_duplicate_transaction_rows = (
    claims_transactions_cleaned
    .groupBy(
        *claims_transactions_cleaned.columns
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

exact_duplicate_group_count = (
    exact_duplicate_transaction_rows.count()
)

exact_duplicate_extra_row_count = (
    exact_duplicate_transaction_rows
    .select(
        F.coalesce(
            F.sum(
                F.col("count") - 1
            ),
            F.lit(0)
        ).alias(
            "duplicate_extra_rows"
        )
    )
    .first()["duplicate_extra_rows"]
)

# Check transaction primary-key uniqueness
duplicate_transaction_ids = (
    claims_transactions_silver
    .groupBy(
        "transaction_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_transaction_id_count = (
    duplicate_transaction_ids.count()
)

# Profile transactions per claim
transactions_per_claim = (
    claims_transactions_silver
    .groupBy(
        "claim_id"
    )
    .agg(
        F.count("*").alias(
            "transaction_count"
        )
    )
)

transaction_distribution = (
    transactions_per_claim
    .agg(
        F.min("transaction_count").alias(
            "minimum_transactions_per_claim"
        ),
        F.avg("transaction_count").alias(
            "average_transactions_per_claim"
        ),
        F.max("transaction_count").alias(
            "maximum_transactions_per_claim"
        )
    )
    .first()
)

median_transactions_per_claim = (
    transactions_per_claim
    .approxQuantile(
        "transaction_count",
        [0.5],
        0.001
    )[0]
)

print(
    f"Exact duplicate transaction groups: "
    f"{exact_duplicate_group_count:,}"
)

print(
    f"Extra exact duplicate transaction rows: "
    f"{exact_duplicate_extra_row_count:,}"
)

print(
    f"Duplicated transaction IDs: "
    f"{duplicate_transaction_id_count:,}"
)

print(
    f"Minimum transactions per claim: "
    f"{transaction_distribution['minimum_transactions_per_claim']:,}"
)

print(
    f"Median transactions per claim: "
    f"{median_transactions_per_claim:,.0f}"
)

print(
    f"Average transactions per claim: "
    f"{transaction_distribution['average_transactions_per_claim']:.3f}"
)

print(
    f"Maximum transactions per claim: "
    f"{transaction_distribution['maximum_transactions_per_claim']:,}"
)

if exact_duplicate_group_count > 0:
    display(
        exact_duplicate_transaction_rows.limit(20)
    )

    raise ValueError(
        "Exact duplicate transaction records were detected."
    )

if duplicate_transaction_id_count > 0:
    display(
        duplicate_transaction_ids.limit(20)
    )

    raise ValueError(
        "Duplicate transaction IDs were detected."
    )

print(
    "No exact duplicate transaction rows were detected."
)

print(
    "transaction_id passed the primary-key "
    "uniqueness validation."
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 40, Finished, Available, Finished, False)

Exact duplicate transaction groups: 0
Extra exact duplicate transaction rows: 0
Duplicated transaction IDs: 0
Minimum transactions per claim: 1
Median transactions per claim: 4
Average transactions per claim: 6.033
Maximum transactions per claim: 130
No exact duplicate transaction rows were detected.
transaction_id passed the primary-key uniqueness validation.


In [39]:
# ============================================================
# 9.1 VALIDATE PAYMENT-AMOUNT SOURCE LINEAGE
# ============================================================

# Recreate the expected payment value directly from
# the cleaned source PAYMENTS column
expected_payment_values = (
    claims_transactions_cleaned
    .select(
        "transaction_id",
        F.col("payment_amount")
        .cast(
            T.DecimalType(18, 2)
        )
        .alias(
            "expected_payment_amount_from_payments"
        )
    )
)

payment_lineage_comparison = (
    claims_transactions_silver.alias("silver")
    .select(
        "transaction_id",
        "transaction_type",
        "transaction_amount",
        "payment_amount"
    )
    .join(
        expected_payment_values.alias("source"),
        F.col("silver.transaction_id")
        ==
        F.col("source.transaction_id"),
        "inner"
    )
)

# Null-safe comparison treats two null values as matching
payment_lineage_mismatches = (
    payment_lineage_comparison
    .filter(
        ~F.col("silver.payment_amount").eqNullSafe(
            F.col(
                "source.expected_payment_amount_from_payments"
            )
        )
    )
)

payment_lineage_mismatch_count = (
    payment_lineage_mismatches.count()
)

# Show why AMOUNT cannot be substituted for PAYMENTS
payment_rows_with_null_transaction_amount = (
    payment_lineage_comparison
    .filter(
        (F.col("transaction_type") == "PAYMENT")
        &
        F.col("transaction_amount").isNull()
        &
        F.col("payment_amount").isNotNull()
    )
    .count()
)

print(
    f"Payment-lineage mismatches: "
    f"{payment_lineage_mismatch_count:,}"
)

print(
    f"Payment rows with null AMOUNT but populated PAYMENTS: "
    f"{payment_rows_with_null_transaction_amount:,}"
)

if payment_lineage_mismatch_count > 0:
    display(
        payment_lineage_mismatches.limit(20)
    )

    raise ValueError(
        "Silver payment amounts do not match "
        "the source PAYMENTS column."
    )

print(
    "Payment lineage validation passed."
)

print(
    "Silver payment_amount is sourced from PAYMENTS, "
    "not AMOUNT."
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 41, Finished, Available, Finished, False)

Payment-lineage mismatches: 0
Payment rows with null AMOUNT but populated PAYMENTS: 252,335
Payment lineage validation passed.
Silver payment_amount is sourced from PAYMENTS, not AMOUNT.


In [40]:
# ============================================================
# 10. ADD RECORD HASH AND AUDIT METADATA
# ============================================================

# Include all transformed business and quality columns
# in the deterministic record hash
transaction_hash_columns = (
    claims_transactions_silver.columns
)

transaction_hash_inputs = [
    F.coalesce(
        F.col(column_name).cast("string"),
        F.lit("<NULL>")
    )
    for column_name in transaction_hash_columns
]

claims_transactions_silver_final = (
    claims_transactions_silver

    # Generate a SHA-256 hash of the Silver business values
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *transaction_hash_inputs
            ),
            256
        )
    )

    # Add data-lineage metadata
    .withColumn(
        "source_system",
        F.lit(SOURCE_SYSTEM)
    )
    .withColumn(
        "source_file",
        F.lit(SOURCE_FILE)
    )
    .withColumn(
        "silver_processed_timestamp",
        F.current_timestamp()
    )
)

print(
    f"Hash input columns: "
    f"{len(transaction_hash_columns)}"
)

print(
    f"Final Silver columns: "
    f"{len(claims_transactions_silver_final.columns)}"
)

claims_transactions_silver_final.select(
    "transaction_id",
    "claim_id",
    "transaction_type",
    "transaction_amount",
    "payment_amount",
    "service_interval_quality",
    "record_hash",
    "source_system",
    "source_file",
    "silver_processed_timestamp"
).show(
    5,
    truncate=False
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 42, Finished, Available, Finished, False)

Hash input columns: 41
Final Silver columns: 45
+------------------------------------+------------------------------------+----------------+------------------+--------------+------------------------+----------------------------------------------------------------+-------------------------+-----------------------+--------------------------+
|transaction_id                      |claim_id                            |transaction_type|transaction_amount|payment_amount|service_interval_quality|record_hash                                                     |source_system            |source_file            |silver_processed_timestamp|
+------------------------------------+------------------------------------+----------------+------------------+--------------+------------------------+----------------------------------------------------------------+-------------------------+-----------------------+--------------------------+
|5c58de0b-3ba9-401d-2ce7-a4785ff35c15|e413105d-6f23-34ef-724a-b42adab9

In [41]:
# ============================================================
# 10.1 VALIDATE RECORD HASHES AND AUDIT METADATA
# ============================================================

transaction_hash_validation = (
    claims_transactions_silver_final
    .agg(
        F.count("*").alias(
            "final_row_count"
        ),

        F.countDistinct(
            "transaction_id"
        ).alias(
            "final_distinct_transaction_ids"
        ),

        F.sum(
            F.when(
                F.col("record_hash").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_record_hashes"
        ),

        F.sum(
            F.when(
                F.length("record_hash") != 64,
                1
            ).otherwise(0)
        ).alias(
            "invalid_hash_lengths"
        ),

        F.sum(
            F.when(
                F.col("source_system").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_source_systems"
        ),

        F.sum(
            F.when(
                F.col("source_file").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_source_files"
        ),

        F.sum(
            F.when(
                F.col(
                    "silver_processed_timestamp"
                ).isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_processed_timestamps"
        )
    )
)

display(
    transaction_hash_validation
)

transaction_hash_result = (
    transaction_hash_validation.first()
)

assert transaction_hash_result[
    "final_row_count"
] == 711238, (
    "The final transaction row count is incorrect."
)

assert transaction_hash_result[
    "final_distinct_transaction_ids"
] == 711238, (
    "The final transaction IDs are not unique."
)

assert transaction_hash_result[
    "null_record_hashes"
] == 0, (
    "Null transaction record hashes were detected."
)

assert transaction_hash_result[
    "invalid_hash_lengths"
] == 0, (
    "Invalid SHA-256 hash lengths were detected."
)

assert transaction_hash_result[
    "null_source_systems"
] == 0, (
    "Null source-system metadata was detected."
)

assert transaction_hash_result[
    "null_source_files"
] == 0, (
    "Null source-file metadata was detected."
)

assert transaction_hash_result[
    "null_processed_timestamps"
] == 0, (
    "Null Silver processing timestamps were detected."
)

print(
    "Transaction record-hash and audit-metadata "
    "validations passed."
)

print(
    f"Final transactions ready for writing: "
    f"{transaction_hash_result['final_row_count']:,}"
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2d7c2ab5-79d9-44cd-a754-03244f58c9aa)

Transaction record-hash and audit-metadata validations passed.
Final transactions ready for writing: 711,238


In [42]:
# ============================================================
# 12. VERIFY THE SAVED SILVER CLAIM-TRANSACTIONS TABLE
# ============================================================

saved_claim_transactions = (
    spark.table(
        TARGET_TABLE
    )
)

saved_transaction_validation = (
    saved_claim_transactions
    .agg(
        F.count("*").alias(
            "saved_row_count"
        ),

        F.countDistinct(
            "transaction_id"
        ).alias(
            "saved_distinct_transaction_ids"
        ),

        F.countDistinct(
            "claim_id"
        ).alias(
            "saved_distinct_claim_ids"
        ),

        F.sum(
            F.when(
                F.col("record_hash").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "saved_null_hashes"
        ),

        F.sum(
            F.when(
                F.col("transaction_type") == "PAYMENT",
                1
            ).otherwise(0)
        ).alias(
            "saved_payment_rows"
        ),

        F.sum(
            F.when(
                F.col("service_interval_quality")
                ==
                "epoch placeholder normalized to null",
                1
            ).otherwise(0)
        ).alias(
            "saved_epoch_placeholders"
        ),

        F.sum(
            F.when(
                F.col("service_interval_quality")
                ==
                "end before start normalized to null",
                1
            ).otherwise(0)
        ).alias(
            "saved_invalid_intervals"
        )
    )
)

display(
    saved_transaction_validation
)

saved_transaction_result = (
    saved_transaction_validation.first()
)

assert saved_transaction_result[
    "saved_row_count"
] == 711238

assert saved_transaction_result[
    "saved_distinct_transaction_ids"
] == 711238

assert saved_transaction_result[
    "saved_distinct_claim_ids"
] == 117889

assert saved_transaction_result[
    "saved_null_hashes"
] == 0

assert saved_transaction_result[
    "saved_payment_rows"
] == 252335

assert saved_transaction_result[
    "saved_epoch_placeholders"
] == 71066

assert saved_transaction_result[
    "saved_invalid_intervals"
] == 22

print(
    "Silver claim-transactions table verification passed."
)

print(
    f"Saved transactions: "
    f"{saved_transaction_result['saved_row_count']:,}"
)

print(
    f"Unique transaction IDs: "
    f"{saved_transaction_result['saved_distinct_transaction_ids']:,}"
)

print(
    f"Claims represented: "
    f"{saved_transaction_result['saved_distinct_claim_ids']:,}"
)

print(
    f"Payment transactions: "
    f"{saved_transaction_result['saved_payment_rows']:,}"
)

display(
    saved_claim_transactions.select(
        "transaction_id",
        "claim_id",
        "patient_id",
        "encounter_id",
        "transaction_type",
        "transaction_amount",
        "payment_amount",
        "service_interval_quality",
        "record_hash",
        "source_file",
        "silver_processed_timestamp"
    ).limit(10)
)

StatementMeta(, f38de220-1996-41d9-9cb1-a5a89050af70, 44, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 275ed7f8-9ce5-473f-95ac-1da52de8c895)

Silver claim-transactions table verification passed.
Saved transactions: 711,238
Unique transaction IDs: 711,238
Claims represented: 117,889
Payment transactions: 252,335


SynapseWidget(Synapse.DataFrame, eeee4da1-6dcf-4731-912d-ff35d4542894)